In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(123)

class DummyNeuNet(nn.Module):
    def __init__(self, in_dim: int = 8, out_dim: int = 2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 30)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(30, 15)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(15, out_dim)

    def forward(self, x):
        z1 = self.fc1(x)
        h1 = self.relu1(z1)

        z2 = self.fc2(h1)
        h2 = self.relu2(z2)

        logits = self.fc3(h2)
        return {
            "z1": z1,
            "h1": h1,
            "z2": z2,
            "h2": h2,
            "logits": logits
        }

model = DummyNeuNet()
print(model)

model_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total trainable params: ", model_params)
print(model.fc2.weight)


DummyNeuNet(
  (fc1): Linear(in_features=8, out_features=30, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=30, out_features=15, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=15, out_features=2, bias=True)
)
Total trainable params:  767
Parameter containing:
tensor([[-0.0666,  0.0943,  0.0573, -0.0473, -0.0500, -0.1615, -0.0500, -0.0740,
         -0.0995, -0.1649,  0.1430, -0.1632,  0.1812, -0.0958, -0.0140,  0.1489,
          0.0602, -0.0521, -0.1470, -0.0746,  0.1470, -0.0689,  0.1522, -0.0314,
         -0.0233,  0.0729, -0.0268, -0.0015,  0.1264,  0.0610],
        [-0.0073,  0.0695,  0.1590,  0.0460, -0.0535,  0.0598, -0.0160, -0.1427,
         -0.0705,  0.0830,  0.0060,  0.0674, -0.1069,  0.1726, -0.0762,  0.0389,
         -0.0892, -0.0881,  0.0818, -0.0510, -0.1158, -0.0746,  0.1331,  0.1099,
          0.1111, -0.1558,  0.0860,  0.0456, -0.1228,  0.0058],
        [ 0.0365, -0.0986, -0.0770,  0.1489, -0.0148, -0.0019, -0.1156, -0.1092,
          0.1681,  0.0679

In [6]:
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(123)

X_train = torch.rand(5, 8)
y_train = torch.tensor([0, 0, 1, 1, 0])

X_test = torch.rand(3, 8)
y_test = torch.tensor([1, 1, 0])

class DummyDataset(Dataset):
    def __init__(self, X, y):
        super().__init__()
        self.features = X
        self.labels = y

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]
    
    def __len__(self):
        return self.labels.shape[0]
        
train_ds = DummyDataset(X_train, y_train)
test_ds = DummyDataset(X_test, y_test)

train_dataloader = DataLoader(
    train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0
)

test_dataloader = DataLoader(
    test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

print(len(train_dataloader))

for idx, (features, labels) in enumerate(train_dataloader):
    print(f"batch {idx+1}", (features, labels))

3
batch 1 (tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740, 0.8665, 0.1366, 0.1025],
        [0.1186, 0.8274, 0.3821, 0.6605, 0.8536, 0.5932, 0.6367, 0.9826]]), tensor([0, 1]))
batch 2 (tensor([[0.2745, 0.6584, 0.2775, 0.8573, 0.8993, 0.0390, 0.9268, 0.7388],
        [0.7179, 0.7058, 0.9156, 0.4340, 0.0772, 0.3565, 0.1479, 0.5331]]), tensor([1, 0]))
batch 3 (tensor([[0.1841, 0.7264, 0.3153, 0.6871, 0.0756, 0.1966, 0.3164, 0.4017]]), tensor([0]))


In [7]:
import torch

torch.manual_seed(123)

model = DummyNeuNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
epochs = 3

for ep in range(epochs):
    model.train()

    for idx, (features, labels) in enumerate(train_dataloader):
        output = model(features)
        logits = output["logits"]
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {ep+1:03d}/{epochs:03d}, Batch {idx+1:03d}/{len(train_dataloader):03d}"
            f"train loss: {loss:0.4f}")

    model.eval()


Epoch 001/003, Batch 003/003train loss: 1.0274
Epoch 002/003, Batch 003/003train loss: 0.3583
Epoch 003/003, Batch 003/003train loss: 0.6188


In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(X_train)

print(outputs["logits"])
probs = F.softmax(outputs["logits"], dim=1)
print(probs)
predictions = torch.argmax(probs, dim=1)
print(predictions)

def compute_accuracy(model, dataloader):
    model.eval()
    total_samples = 0
    correct = 0

    for idx, (features, labels) in enumerate(dataloader):
        with torch.no_grad():
            
        predictions = torch.argmax(logits, dim=1)
        correct += torch.sum( (predictions == labels) )
        total_samples += len(labels)

    return (correct / total_samples).item()

print(compute_accuracy(model, train_dataloader))
print(compute_accuracy(model, test_dataloader))


tensor([[ 0.7279, -0.1518],
        [ 0.2616,  0.2640],
        [-1.0924,  1.3343],
        [-1.2554,  1.5082],
        [ 0.7871, -0.1919]])
tensor([[0.7068, 0.2932],
        [0.4994, 0.5006],
        [0.0812, 0.9188],
        [0.0593, 0.9407],
        [0.7269, 0.2731]])
tensor([0, 1, 1, 1, 0])


TypeError: argmax(): argument 'input' (position 1) must be Tensor, not dict

In [ ]:
model.eval()
outputs = model(X_train)
print(outputs.keys())

for name, value in outputs.items():
    print(name, value.shape)

print(outputs["h1"][0].shape)
print(outputs["h1"][0])     # activation of sample 0
print(outputs["h1"][:, 3])  # activation of 3rd neurons for all samples
print(outputs["h1"][3, : ])  # activation of all 30 neurons for 3rd sample

print(outputs["z1"][0])
print(outputs["h1"][0]) 

h1 = outputs["h1"]
mean_activation = h1.mean(dim=0)
print("Mean actviation shape:  ", mean_activation.shape)
print(mean_activation)

In [ ]:
values, indices = torch.sort(       # sort return two lists: values, indicies
    mean_activation,
    descending=True
)
print(values, "\n", indices)

print("Neuron Ranking")
for rank, (neuron_idx, score) in enumerate(zip(indices, values), start=1):
    print(f"Rank {rank:02d} | Neuron {neuron_idx.item():02d} | Mean activation {score.item():0.4f}")

activation_frequency = (
    h1 > 0
).float().mean(dim=0)
print(activation_frequency)

max_activation = h1.max(dim=0).values
print(max_activation)

In [ ]:
for neuron_idx in range(h1.shape[1]):

    print(
        f"Neuron {neuron_idx:02d} | "
        f"mean={mean_activation[neuron_idx]:.4f} | "
        f"freq={activation_frequency[neuron_idx]:.2f} | "
        f"max={max_activation[neuron_idx]:.4f}"
    )